## Práctica, Aplicaciones con Hugging Face y Gradio

En el siguiente proyecto vamos a utilizar los modelos existentes en la libreria de Hugging Face para crear una aplicación para realizar tareas de ASR (reconocimiento automático del habla), también generaremos una interfaz de usuario mediante la libreria de Gradio.

En primer lugar cargamos nuestra API-Key de Hugging face para poder trabajar con sus modelos.

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
hf_token = os.getenv("HF_TOKEN")

print(f"Token loaded: {hf_token is not None}")

### Separación de las voces de la instrumental

Se realiza una separación de la pista sobre la que vamos a trabajar con el objetivo de obtener mejores resultados en el reconocimiento automático del habla. Para ello hacemos uso del modelo open source Demucs y almacenamos los output de audio en ficheros utilizando la librería soundfile.

In [ ]:
import sys
import os
from demucs import pretrained
from demucs.apply import apply_model
import torchaudio
import torch

def separate_audio(input_path: str = "audio\Duvet.mp3", output_dir: str = "stems"):
    """
    Separates an input song into stems using Demucs.
    Stems: vocals, drums, bass, other.
    """
    # Crea el directorio de pistas si no existe
    os.makedirs(output_dir, exist_ok=True)

    # Cargar modelo preentrenado Demucs
    model = pretrained.get_model('htdemucs') 
    # Seteo el modelo en modo evaluación (desactiva comportamientos de entrenamiento) 
    model.eval()

    # Cargar fichero de audio
    wav, sr = torchaudio.load(input_path)
    # Evito conversión a mono - mantengo canales originales para Demucs
    wav = wav.to(torch.device('cuda' if torch.cuda.is_available() else 'cpu'))
   
    model = model.to(wav.device)

    # Corro el modelo sobre el audio
    with torch.no_grad():
        estimates = apply_model(model, wav[None], split=True, overlap=0.25)[0]

    # Escribir las pistas en archivos separados
    for source, audio in zip(model.sources, estimates):
        output_path = os.path.join(output_dir, f"{source}.mp3")
        torchaudio.save(output_path, audio.cpu(), sample_rate=sr)
        print(f"Fichero guardado: {output_path}")

    print("\n Separación exitosa. Las pistas de audio se encuentran en:", os.path.abspath(output_dir))

# Execute the separation
separate_audio()

#### Tarea de ASR con Whisper en local

Uso del modelo de la librería de Hugging Face en local a través de un pipeline para realizar tareas de ASR con canciones. Nuestro objetivo en este caso es obtener la una transcripción de la letra de la canción y almacenarla en un fichero de texto con el que trabajaremos posteriormente.

In [ ]:
import os
import librosa
from transformers import pipeline
import torch

# --- Configuración ---
AUDIO_FILE_PATH = os.path.join("stems", "vocals.mp3")
TRANSCRIPTION_DIR = "transcriptions"
TRANSCRIPTION_FILE_PATH = os.path.join(TRANSCRIPTION_DIR, "transcription_with_timestamps_duvet.txt")

# --- 1. Verificar Configuración y Cargar Modelo ---
print("--- Script de Transcripción ASR ---")

# Comprobar si la GPU está disponible
if torch.cuda.is_available():
    print(f"GPU disponible. Usando dispositivo: {torch.cuda.get_device_name(0)}")
    device = 0
else:
    print("GPU no encontrada. Usando CPU en su lugar. Esto será lento.")
    device = -1

# Cargar el pipeline de ASR
print("\nCargando modelo Whisper... (Esto puede tardar un momento)")
try:
    pipe = pipeline(
        "automatic-speech-recognition", 
        model="openai/whisper-large-v3",
        device=device,
        dtype=torch.float16)
    print("Modelo cargado con éxito.")
except Exception as e:
    print(f"Error al cargar el modelo: {e}")
    # Salir de la celda si el modelo no se puede cargar
    exit()

# --- 2. Cargar y Transcribir Audio ---
print(f"\nProcesando archivo de audio: {AUDIO_FILE_PATH}")

# Comprobar si el archivo de audio existe
if not os.path.exists(AUDIO_FILE_PATH):
    print(f"Error: No se encontró el archivo de audio en '{AUDIO_FILE_PATH}'")
else:
    try:
        # Cargar audio usando librosa
        audio, sr = librosa.load(AUDIO_FILE_PATH, sr=16000)
        
        print("Transcribiendo audio... (Esto puede tardar dependiendo de la duración de la pista)")
        result = pipe(audio, return_timestamps=True, generate_kwargs={"task": "transcribe", "language": "en"})
        print("Transcripción completada.")

        # --- 3. Guardar y Mostrar Resultados ---
        
        # Asegurarse de que exista el directorio de salida 
        os.makedirs(TRANSCRIPTION_DIR, exist_ok=True)
        
        
        # Guardar la transcripción detallada con marcas de tiempo
        with open(TRANSCRIPTION_FILE_PATH, "w", encoding="utf-8") as f:
            f.write("--- Transcripción Completa ---\n")
            f.write(result['text'].strip() + "\n\n")
            f.write("--- Segmentos con Marcas de Tiempo ---\n")
            for chunk in result['chunks']:
                start_time = round(chunk['timestamp'][0], 2)
                end_time = round(chunk['timestamp'][1], 2)
                start_formatted = f"{int(start_time//60):02d}:{int(start_time%60):02d}"
                end_formatted = f"{int(end_time//60):02d}:{int(end_time%60):02d}"
                f.write(f"[{start_formatted} -> {end_formatted}] {chunk['text'].strip()}\n")
        
        print(f"\nTranscripción guardada en: {TRANSCRIPTION_FILE_PATH}")
        
        # Imprimir el texto final de la transcripción en la consola
        print("\n--- Resultado de la Transcripción ---")
        print(result['text'])

    except Exception as e:
        print(f"Ocurrió un error durante el procesamiento o la transcripción del audio: {e}")


###  DIARIZACIÓN (identificar hablantes)
Diarización de Voces (Detección de Cantantes o Hablantes)
 Analiza la pista de voz separada y detecta cuántos cantantes/hablantes hay, 
 generando un archivo RTTM con los segmentos y mostrando por consola los intervalos de cada voz.





In [1]:
import os
import torch
from pyannote.audio import Pipeline


DIARIZATION_AUDIO = "stems/vocals.mp3"
DIARIZATION_OUTPUT = "diarization/diarization_output.rttm"

print("\n--- Iniciando diarización de voces ---")

try:
    diarization_pipeline = Pipeline.from_pretrained(
        "pyannote/speaker-diarization-3.1",
        use_auth_token=os.getenv("HF_TOKEN")
    )
    if torch.cuda.is_available():
        diarization_pipeline.to(torch.device("cuda"))
        print("GPU detectada y usada para diarización.")
    else:
        print("Usando CPU para diarización (más lento).")

    # Ejecutar diarización
    diarization = diarization_pipeline(DIARIZATION_AUDIO)

    os.makedirs("diarization", exist_ok=True)
    with open(DIARIZATION_OUTPUT, "w") as f:
        diarization.write_rttm(f)

    print(f"✅ Diarización completada y guardada en: {DIARIZATION_OUTPUT}")

except Exception as e:
    print(f"❌ Error en la diarización: {e}")




c:\Users\bryan\Desktop\proyectosClase\Trabajo-IA-1\.venv\Lib\site-packages\pyannote\audio\core\io.py:43: UserWarning: torchaudio._backend.set_audio_backend has been deprecated. With dispatcher enabled, this function is no-op. You can remove the function call.
  torchaudio.set_audio_backend("soundfile")



--- Iniciando diarización de voces ---


c:\Users\bryan\Desktop\proyectosClase\Trabajo-IA-1\.venv\Lib\site-packages\pyannote\audio\pipelines\speaker_verification.py:43: UserWarning: torchaudio._backend.get_audio_backend has been deprecated. With dispatcher enabled, this function is no-op. You can remove the function call.
  backend = torchaudio.get_audio_backend()
c:\Users\bryan\Desktop\proyectosClase\Trabajo-IA-1\.venv\Lib\site-packages\pyannote\audio\pipelines\speaker_verification.py:45: UserWarning: Module 'speechbrain.pretrained' was deprecated, redirecting to 'speechbrain.inference'. Please update your script. This is a change from SpeechBrain 1.0. See: https://github.com/speechbrain/speechbrain/releases/tag/v1.0.0
  from speechbrain.pretrained import (
c:\Users\bryan\Desktop\proyectosClase\Trabajo-IA-1\.venv\Lib\site-packages\pyannote\audio\pipelines\speaker_verification.py:53: UserWarning: torchaudio._backend.set_audio_backend has been deprecated. With dispatcher enabled, this function is no-op. You can remove the func

GPU detectada y usada para diarización.


c:\Users\bryan\Desktop\proyectosClase\Trabajo-IA-1\.venv\Lib\site-packages\pyannote\audio\utils\reproducibility.py:74: ReproducibilityWarning: TensorFloat-32 (TF32) has been disabled as it might lead to reproducibility issues and lower accuracy.
It can be re-enabled by calling
   >>> import torch
   >>> torch.backends.cuda.matmul.allow_tf32 = True
   >>> torch.backends.cudnn.allow_tf32 = True
See https://github.com/pyannote/pyannote-audio/issues/1370 for more details.

  warnings.warn(
c:\Users\bryan\Desktop\proyectosClase\Trabajo-IA-1\.venv\Lib\site-packages\pyannote\audio\models\blocks\pooling.py:104: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\ReduceOps.cpp:1823.)
  std = sequences.std(dim=-1, correction=1)
c:\Users\bryan\Desktop\proyectosClase\Trabajo-IA-1\.venv\Lib\site-

✅ Diarización completada y guardada en: diarization/diarization_output.rttm


### Fusión de Transcripción y Diarización 
 Combina los resultados de Whisper (transcripción con timestamps)
 y Pyannote (diarización) para mostrar qué dijo cada cantante.

In [3]:
import os, json

t_file = "transcriptions/transcription_with_timestamps.txt"
d_file = "diarization/diarization_output.rttm"
out_txt = "fusions/transcription_with_speakers.txt"
out_json = "fusions/transcription_with_speakers.json"

# --- Leer diarización ---
diar = []
for l in open(d_file, encoding="utf-8"):
    p = l.split()
    if len(p) >= 8:
        s, e, spk = float(p[3]), float(p[3]) + float(p[4]), p[7].upper().replace("SPEAKER", "SPEAKER_").replace("__", "_")
        diar.append((s, e, spk))

# --- Leer transcripción ---
def sec(t): m, s = t.split(":"); return int(m)*60+float(s)
whisper = []
for l in open(t_file, encoding="utf-8"):
    if "[" in l and "->" in l:
        tp, tx = l.split("]")
        st, en = [x.strip() for x in tp.strip("[]").split("->")]
        whisper.append((sec(st), sec(en), tx.strip()))

# --- Fusión ---
def split(ws, we, tx):
    sub = [(max(ws, ds), min(we, de), tx, spk) for ds, de, spk in diar if min(we, de) > max(ws, ds)]
    return sub or [(ws, we, tx, "UNKNOWN")]

comb = sorted([s for w in whisper for s in split(*w)], key=lambda x: x[0])

merged, last = [], None
for ws, we, tx, spk in comb:
    if last and spk == last[3] and abs(ws - last[1]) < 0.5:
        last = (last[0], we, last[2]+" "+tx, spk)
    else:
        if last: merged.append(last)
        last = (ws, we, tx, spk)
if last: merged.append(last)

# --- Guardar ---
os.makedirs("fusions", exist_ok=True)
with open(out_txt, "w", encoding="utf-8") as f:
    f.writelines(f"[{s}] [{ws:.2f} -> {we:.2f}] {tx}\n" for ws, we, tx, s in merged)
json.dump([{"timestamp":[ws,we],"text":f"{s}: {tx}"} for ws,we,tx,s in merged],
          open(out_json,"w",encoding="utf-8"), indent=2, ensure_ascii=False)






## Gradio

In [ ]:
import gradio as gr
from PIL import Image
import re
import os
import string

# 🔁 mm:ss → segundos (sigue usándose para otros formatos de subtítulos)
def to_seconds(ts):
    mins, secs = map(int, ts.split(":"))
    return mins * 60 + secs

# Normalizar texto para búsquedas (quita puntuación y espacios múltiples)
def _normalize(text):
    t = text.lower()
    # quitar puntuación
    t = t.translate(str.maketrans('', '', string.punctuation))
    # colapsar espacios
    t = re.sub(r"\s+", " ", t).strip()
    return t

# 📖 TXT → subtítulos JSON
# Esta versión:
#  - parsea líneas con formato [SPEAKER_xx] [start -> end] text
#  - detecta segmentos "anidados" (un speaker pequeño dentro de otro más largo)
#    y divide el segmento externo en trozos alrededor del texto del segmento interno
#  - si no puede hacer el split por texto, hace un split temporal aproximado

def load_subtitles(filepath):
    if not os.path.exists(filepath):
        print(f"⚠️ Archivo de subtítulos no encontrado: {filepath}")
        return None

    # Patrón flexible para capturar speaker, start y end con decimales
    pattern = r"\[(?P<speaker>[^\]]+)\]\s*\[(?P<start>\d+(?:\.\d+)?)\s*->\s*(?P<end>\d+(?:\.\d+)?)\]\s*(?P<text>.*)"
    subs_raw = []

    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            m = re.match(pattern, line)
            if m:
                speaker = m.group('speaker')
                start = float(m.group('start'))
                end = float(m.group('end'))
                text = m.group('text').strip()
                subs_raw.append({
                    'speaker': speaker,
                    'start': start,
                    'end': end,
                    'text': text,
                    'norm_text': _normalize(text)
                })
            else:
                # intentar con formato mm:ss -> mm:ss sin speaker
                m2 = re.match(r"\[(?P<start_mm>\d\d:\d\d)\s*->\s*(?P<end_mm>\d\d:\d\d)\]\s*(?P<text_mm>.*)", line)
                if m2:
                    start = to_seconds(m2.group('start_mm'))
                    end = to_seconds(m2.group('end_mm'))
                    text = m2.group('text_mm').strip()
                    subs_raw.append({
                        'speaker': None,
                        'start': float(start),
                        'end': float(end),
                        'text': text,
                        'norm_text': _normalize(text)
                    })
                else:
                    print(f"Línea no parseada: {line}")

    if not subs_raw:
        return None

    # Ordenar por duración (largos primero) para poder dividir los externos usando internos
    subs_raw.sort(key=lambda s: (s['start'], s['end']))

    # Construir estructura para dividir segmentos externos cuando haya internos dentro
    result = []
    used = [False] * len(subs_raw)

    for i, outer in enumerate(subs_raw):
        if used[i]:
            continue
        # encontrar subsegmentos completamente dentro de outer
        nested_idxs = [j for j, s in enumerate(subs_raw) if j != i and not used[j]
                       and s['start'] >= outer['start'] and s['end'] <= outer['end']]
        if not nested_idxs:
            # no nested segments -> añadir tal cual (incluyendo speaker if present)
            entry = {
                'timestamp': [outer['start'], outer['end']],
                'text': f"[{outer['speaker']}] {outer['text']}" if outer['speaker'] else outer['text']
            }
            result.append(entry)
            used[i] = True
            continue

        # procesar outer dividido por nested (ordenar nested por start)
        nested_idxs.sort(key=lambda j: subs_raw[j]['start'])
        remaining_text = outer['text']
        remaining_norm = outer['norm_text']
        cursor_start = outer['start']
        pieces = []

        # intento secuencial de encontrar los textos nested dentro del outer
        fail_find = False
        for j in nested_idxs:
            nested = subs_raw[j]
            # buscar nested.norm_text dentro de remaining_norm
            idx = remaining_norm.find(nested['norm_text'])
            if idx >= 0:
                # determinar split positions on original remaining_text using a regex search for approximate substring
                # usamos re.escape para buscar la primera ocurrencia ignorando case and punctuation differences
                pattern_search = re.escape(nested['text'][:50])  # buscar primer trozo representativo
                msearch = re.search(pattern_search, remaining_text, flags=re.IGNORECASE)
                if msearch:
                    pos = msearch.start()
                    before_text = remaining_text[:pos].strip()
                    # advance remaining_text
                    remaining_text = remaining_text[pos + len(msearch.group(0)):].strip()
                    # actualizar remaining_norm aproximando
                    remaining_norm = _normalize(remaining_text)

                    # piece before nested
                    if before_text:
                        pieces.append({'start': cursor_start, 'end': nested['start'], 'text': before_text})
                    # nested itself
                    pieces.append({'start': nested['start'], 'end': nested['end'], 'text': f"[{nested['speaker']}] {nested['text']}" if nested['speaker'] else nested['text']})
                    cursor_start = nested['end']
                    used[j] = True
                else:
                    # no encontramos la subcadena en la forma esperada -> fallback temporal split
                    fail_find = True
                    break
            else:
                fail_find = True
                break

        if fail_find:
            # Fallback: repartir el texto de outer proporcionalmente entre intervals (outer and nested)
            # construiremos intervalos ordenados: outer.start, nested.starts/ends, outer.end
            boundaries = [outer['start']]
            for j in nested_idxs:
                boundaries.append(subs_raw[j]['start'])
                boundaries.append(subs_raw[j]['end'])
            boundaries.append(outer['end'])
            boundaries = sorted(set(boundaries))

            # para cada subinterval, asignar el speaker con menor duración que cubre ese intervalo (priorizar nested)
            for k in range(len(boundaries)-1):
                a = boundaries[k]
                b = boundaries[k+1]
                # encontrar candidatos que cover [a,b)
                candidates = [s for s in subs_raw if s['start'] <= a and s['end'] >= b]
                if candidates:
                    # elegir candidato más corto (inner)
                    cand = min(candidates, key=lambda s: s['end']-s['start'])
                    text_label = f"[{cand['speaker']}]" if cand['speaker'] else ''
                else:
                    text_label = f"[{outer['speaker']}]" if outer['speaker'] else ''
                # asignar una porción del texto según proporción temporal
                total_len = len(outer['text'])
                # safe guard
                if total_len == 0:
                    part_text = ''
                else:
                    # compute proportional substring
                    rel_start = (a - outer['start']) / (outer['end'] - outer['start'])
                    rel_end = (b - outer['start']) / (outer['end'] - outer['start'])
                    sidx = int(rel_start * total_len)
                    eidx = int(rel_end * total_len)
                    part_text = outer['text'][sidx:eidx].strip()
                if part_text:
                    result.append({'timestamp':[a,b],'text': f"{text_label} {part_text}".strip()})
            used[i] = True
            # mark nested used
            for j in nested_idxs:
                used[j] = True
            continue

        # si llegamos aquí, usamos 'pieces' que ya contiene before, nested, etc.
        # agregar pieza final si queda texto
        if remaining_text:
            pieces.append({'start': cursor_start, 'end': outer['end'], 'text': remaining_text})

        # agregar piezas al resultado
        for p in pieces:
            if p['text'].strip():
                # si p['text'] ya contiene speaker tag (como en nested), queda así
                if p['text'].startswith('['):
                    text_out = p['text']
                else:
                    text_out = f"[{outer['speaker']}] {p['text']}" if outer['speaker'] else p['text']
                result.append({'timestamp':[float(p['start']), float(p['end'])],'text': text_out})

        used[i] = True

    # por seguridad, añadir cualquier segmento no usado aún
    for idx, s in enumerate(subs_raw):
        if not used[idx]:
            result.append({'timestamp':[s['start'], s['end']],'text': f"[{s['speaker']}] {s['text']}" if s['speaker'] else s['text']})

    # ordenar resultado por timestamp
    result.sort(key=lambda x: x['timestamp'][0])

    print(f"Cargados {len(result)} subtítulos (procesados).")
    return result if result else None

# 🎵 Canciones
songs = {
    "Bring Me To Life — Evanescence": {
        "audio": "audio/bring-me-to-life-evanescence.mp3",
        "cover": "covers/bringmeto.jpg",
        "subtitles": "transcriptions/transcription_with_speakers.txt"
    },
    "Duvet — Boa": {
        "audio": "audio/Duvet.mp3",
        "cover": "covers/duvet.jpg",
        "subtitles": "transcriptions/transcription_with_timestamps_duvet.txt"
    }
}

# 🔄 Actualizar reproductor
def update_player(song_name):
    song = songs[song_name]

    cover_img = Image.open(song["cover"])
    subs = load_subtitles(song["subtitles"])
    
    if subs:
        print(f"Subtítulos cargados correctamente para {song_name}")
    else:
        print(f"No se pudieron cargar los subtítulos para {song_name}")

    return (
        cover_img,
        gr.update(value=song["audio"], subtitles=subs, autoplay=False),
        f"<div class='songtitle'>{song_name}</div>"
    )

# 🎨 Estilo Spotify
css = """
body { background-color: green; font-family: 'Helvetica', sans-serif; }

#title {
    color:white; font-size:32px; font-weight:bold;
    text-align:center; margin-bottom:20px;
}
.wrapper { background-color: green; padding: 20px; border-radius: 15px; }

.card {
    background: rgba(24,24,24,0.92);
    border: 1px solid rgba(255,255,255,0.08);
    border-radius:25px; padding:28px;
    max-width:450px; margin:auto; text-align:center;
    box-shadow: 0 8px 18px rgba(0,0,0,0.7);
    backdrop-filter: blur(10px);
}

.songtitle {
    color:#1DB954; font-size:22px; font-weight:bold;
    margin-top:14px; margin-bottom:8px;
}

img { border-radius: 15px; transition: .3s; }
img:hover { transform: scale(1.03); box-shadow: 0 6px 18px rgba(0,0,0,0.6); }

.gradio-container .gradio-audio {
    border-radius: 15px;
}
"""

# 🚀 UI
with gr.Blocks(css=css) as demo:
    gr.HTML("<div id='title'>🔥 Música To Guapa</div>")

    with gr.Column(elem_classes="card"):

        song_dropdown = gr.Dropdown(
            choices=list(songs.keys()),
            value=list(songs.keys())[0],
            label="Selecciona una canción"
        )

        cover_output = gr.Image(show_label=False)
        title_output = gr.HTML()
        audio_output = gr.Audio(
            show_label=False,
            autoplay=False,
            interactive=True
        )

        song_dropdown.change(
            fn=update_player,
            inputs=song_dropdown,
            outputs=[cover_output, audio_output, title_output]
        )

        # Inicializar
        init = list(songs.keys())[0]
        cover_val, audio_val, title_val = update_player(init)
        cover_output.value = cover_val
        audio_output.value = songs[init]["audio"]
        audio_output.subtitles = load_subtitles(songs[init]["subtitles"])
        title_output.value = title_val

demo.launch()

Cargados 49 subtítulos (procesados).
Subtítulos cargados correctamente para Bring Me To Life — Evanescence
Cargados 49 subtítulos (procesados).
* Running on local URL:  http://127.0.0.1:7864
* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.
* To create a public link, set `share=True` in `launch()`.


Línea no parseada: --- Transcripción Completa ---
Línea no parseada: And you don't seem to understand A shame you seemed an honest man And all the fears you hold so dear Will turn to whisper in your ear And you know what they say might hurt you And you know that it means too much And you don't even feel a thing I am falling, I am fading I have lost it all And you don't seem the lying kind A shame that I can read your mind And all the things that I read there Candlelit smile, the weep of shame And you know I don't mean to hurt you But you know that it means so much And you don't even feel a thing I am falling, I am fading I am drowning, coming to breathe I am hurting, I have lost it all I am losing, coming to breathe Oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh, oh

GRADOB


In [4]:
import gradio as gr
from PIL import Image
import re, os

# 🔁 mm:ss → segundos
def to_seconds(ts):
    mins, secs = map(int, ts.split(":"))
    return mins * 60 + secs

# 📖 TXT → subtítulos JSON (con color por hablante)
def load_subtitles(filepath):
    pattern_speaker = r"\[(SPEAKER_[0-9]+)\] \[(\d+\.\d+) -> (\d+\.\d+)\] (.*)"
    pattern_basic = r"\[(\d\d:\d\d) -> (\d\d:\d\d)\] (.*)"
    subs = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            m = re.match(pattern_speaker, line)
            if m:
                spk = m.group(1)
                color = "#FFFFFF" if spk == "SPEAKER_00" else "#1DB954"
                text = f"<span style='color:{color}'>{m.group(4)}</span>"
                subs.append({
                    "timestamp": [float(m.group(2)), float(m.group(3))],
                    "speaker": spk,
                    "text": text
                })
                continue
            m = re.match(pattern_basic, line)
            if m:
                text = f"<span style='color:#FFFFFF'>{m.group(3)}</span>"
                subs.append({
                    "timestamp": [to_seconds(m.group(1)), to_seconds(m.group(2))],
                    "speaker": None,
                    "text": text
                })
    return subs

# 🎵 Canciones
songs = {
    "Bring Me To Life — Evanescence": {
        "audio": "audio/bring-me-to-life-evanescence.mp3",
        "cover": "covers/bringmeto.jpg",
        "subtitles": "fusions/transcription_with_speakers.txt"
    },
    "Duvet — Boa": {
        "audio": "audio/Duvet.mp3",
        "cover": "covers/duvet.jpg",
        "subtitles": "fusions/transcription_with_speakers.txt"
    }
}

# 🔄 Actualizar reproductor
def update_player(song_name):
    song = songs[song_name]
    cover_img = Image.open(song["cover"])
    subs = load_subtitles(song["subtitles"])
    return (
        cover_img,
        gr.update(value=song["audio"], subtitles=subs, autoplay=False),
        f"<div class='songtitle'>{song_name}</div>"
    )

# 🎨 Estilo Spotify
css = """
body { background-color: #121212; font-family: 'Helvetica', sans-serif; }
#title { color:white; font-size:32px; font-weight:bold; text-align:center; margin-bottom:20px; }
.card {
    background: rgba(24,24,24,0.92);
    border: 1px solid rgba(255,255,255,0.08);
    border-radius:25px; padding:28px;
    max-width:450px; margin:auto; text-align:center;
    box-shadow: 0 8px 18px rgba(0,0,0,0.7);
    backdrop-filter: blur(10px);
}
.songtitle {
    color:#1DB954; font-size:22px; font-weight:bold;
    margin-top:14px; margin-bottom:8px;
}
img { border-radius: 15px; transition: .3s; }
img:hover { transform: scale(1.03); box-shadow: 0 6px 18px rgba(0,0,0,0.6); }
audio::cue { color: #1DB954; }
"""

# 🚀 UI
with gr.Blocks(css=css) as demo:
    gr.HTML("<div id='title'>🔥 Música To Guapa</div>")

    with gr.Column(elem_classes="card"):
        song_dropdown = gr.Dropdown(
            choices=list(songs.keys()),
            value=list(songs.keys())[0],
            label="Selecciona una canción"
        )

        cover_output = gr.Image(show_label=False)
        title_output = gr.HTML()
        audio_output = gr.Audio(show_label=False)

        song_dropdown.change(
            fn=update_player,
            inputs=song_dropdown,
            outputs=[cover_output, audio_output, title_output]
        )

        # Inicializar
        init = list(songs.keys())[0]
        cover_val, audio_val, title_val = update_player(init)
        cover_output.value = cover_val
        audio_output.value = songs[init]["audio"]
        audio_output.subtitles = load_subtitles(songs[init]["subtitles"])
        title_output.value = title_val

demo.launch()



* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [ ]:
import gradio as gr
from PIL import Image
import re, os

# 🔁 mm:ss → segundos
def to_seconds(ts):
    mins, secs = map(int, ts.split(":"))
    return mins * 60 + secs

# 📖 TXT → subtítulos JSON
def load_subtitles(filepath):
    if not os.path.exists(filepath):
        print(f"⚠️ Archivo de subtítulos no encontrado: {filepath}")
        return None

    pattern = r"\[(\d\d:\d\d) -> (\d\d:\d\d)\] (.*)"
    subs = []

    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            m = re.match(pattern, line.strip())
            if m:
                start = to_seconds(m.group(1))
                end = to_seconds(m.group(2))
                text = m.group(3)

                # 🎨 Detectar hablante
                match = re.match(r"^(SPEAKER_\d+|UNKNOWN):\s*(.*)", text)
                if match:
                    speaker, content = match.groups()
                else:
                    speaker, content = "UNKNOWN", text

                color = {
                    "SPEAKER_00": "#4FC3F7",  # azul
                    "SPEAKER_01": "#F48FB1",  # rosa
                    "UNKNOWN": "#B0BEC5"      # gris
                }.get(speaker, "#FFF")

                # Gradio espera: {"timestamp": [inicio, fin], "text": texto_html}
                subs.append({
                    "timestamp": [float(start), float(end)],
                    "text": f"<span style='color:{color}'><b>{speaker}:</b> {content}</span>"
                })
    return subs if subs else None


# 🎵 Canciones
songs = {
    "Bring Me To Life — Evanescence": {
        "audio": "audio/bring-me-to-life-evanescence.mp3",
        "cover": "covers/bringmeto.jpg",
        "subtitles": "fusions/transcription_with_speakers.txt"
    },
    "Duvet — Boa": {
        "audio": "audio/Duvet.mp3",
        "cover": "covers/duvet.jpg",
        "subtitles": "fusions/transcription_with_timestamps_duvet.txt"
    }
}

# 🔄 Actualizar reproductor
def update_player(song_name):
    song = songs[song_name]
    cover_img = Image.open(song["cover"])
    subs = load_subtitles(song["subtitles"])

    return (
        cover_img,
        gr.update(value=song["audio"], subtitles=subs, autoplay=False),
        f"<div class='songtitle'>{song_name}</div>"
    )


# 🎨 Estilo Spotify
css = """
body { background-color: #121212; font-family: 'Helvetica', sans-serif; }

#title {
    color:white; font-size:32px; font-weight:bold;
    text-align:center; margin-bottom:20px;
}

.card {
    background: rgba(24,24,24,0.92);
    border: 1px solid rgba(255,255,255,0.08);
    border-radius:25px; padding:28px;
    max-width:450px; margin:auto; text-align:center;
    box-shadow: 0 8px 18px rgba(0,0,0,0.7);
    backdrop-filter: blur(10px);
}

.songtitle {
    color:#1DB954; font-size:22px; font-weight:bold;
    margin-top:14px; margin-bottom:8px;
}

img { border-radius: 15px; transition: .3s; }
img:hover { transform: scale(1.03); box-shadow: 0 6px 18px rgba(0,0,0,0.6); }
"""

# 🚀 UI
with gr.Blocks(css=css) as demo:
    gr.HTML("<div id='title'>🔥 Música To Guapa</div>")

    with gr.Column(elem_classes="card"):
        song_dropdown = gr.Dropdown(
            choices=list(songs.keys()),
            value=list(songs.keys())[0],
            label="Selecciona una canción"
        )

        cover_output = gr.Image(show_label=False)
        title_output = gr.HTML()
        audio_output = gr.Audio(show_label=False)

        song_dropdown.change(
            fn=update_player,
            inputs=song_dropdown,
            outputs=[cover_output, audio_output, title_output]
        )

        # Inicializar
        init = list(songs.keys())[0]
        cover_val, audio_val, title_val = update_player(init)
        cover_output.value = cover_val
        audio_output.value = songs[init]["audio"]
        audio_output.subtitles = load_subtitles(songs[init]["subtitles"])
        title_output.value = title_val

demo.launch()
